# Symbolic computing with SymPy

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*X. Cid Vidal* (original material, 2025). Adapted by *J. A. Hernando Morata*.

Derivatives, integrals, series and differential equations: the algebra we do on paper can
be done, exactly, by the computer.

**Objectives**

* understand the difference between **symbolic** (exact) and **numerical** computing;
* create **symbols** and **expressions**, with assumptions, and substitute values;
* simplify, differentiate, integrate, expand in **series** and take limits;
* **solve** equations, systems and ordinary **differential equations**;
* turn an expression into a fast numerical function with **lambdify**, and plot it.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

## 1. Symbolic computing

A numerical library works with floating-point numbers: fast, but approximate. A
**symbolic** library works with mathematical objects: $\sqrt{12}$ stays $2\sqrt{3}$.

* exact arithmetic: integers, rationals, $\pi$, $\sqrt{2}$…
* manipulation of expressions: simplify, expand, factor;
* calculus: derivatives, integrals, series, limits, differential equations.

In [ ]:
print(math.sqrt(12))
print(sp.sqrt(12))
print(sp.Rational(1, 3) + sp.Rational(1, 6))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Exact or approximate?</b></p>

**Decide** what each line prints, before running it.

```python
print(math.sqrt(2) ** 2 == 2)
print(sp.sqrt(2) ** 2 == 2)
```

</div>

In [ ]:
print(math.sqrt(2) ** 2 == 2)
print(sp.sqrt(2) ** 2 == 2)

<details>
<summary><b>Solution</b></summary>

`False` and `True`. `math.sqrt(2)` is a float, `1.4142135623730951`, and its square is
`2.0000000000000004`. `sp.sqrt(2)` is the exact number $\sqrt{2}$, and its square is the
integer `2`.

</details>

### 1.1 SymPy

[SymPy](https://docs.sympy.org/latest/index.html) is the Python library for symbolic
mathematics. It is written in pure Python, and its objects are ordinary Python objects.

We import it as `import sympy as sp`, so that it is always clear which `sqrt`, `sin` or
`exp` we use: `math.sin` and `np.sin` work on numbers; `sp.sin` works on symbols.

## 2. Symbols and expressions

### 2.1 Symbols

A **symbol** (class `Symbol`) is a mathematical variable. Operations on symbols build
**expressions**:

In [ ]:
x, y = sp.symbols('x y')
expr = x**2 + y**2
expr

In [ ]:
side = sp.Symbol('side')
print(side**3)
print(sp.symbols('m1:4'))

* `'m1:4'` is a range: it creates `m1`, `m2`, `m3`.
* The Python name and the symbol name are independent, but keep them equal: it avoids
  confusion.

### 2.2 Assumptions

A symbol can carry **assumptions**: `positive`, `real`, `integer`… SymPy only applies
a rule if it is true for every value the symbol can take.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Is the square root of a square the number itself?</b></p>

**Decide** what each line prints, before running it.

```python
r = sp.symbols('r', positive=True)
print(sp.sqrt(x**2))
print(sp.sqrt(r**2))
```

</div>

In [ ]:
r = sp.symbols('r', positive=True)
print(sp.sqrt(x**2))
print(sp.sqrt(r**2))

<details>
<summary><b>Solution</b></summary>

`sqrt(x**2)` and `r`. For a generic (complex) `x`, $\sqrt{x^2} \neq x$ (try $x=-1$),
so SymPy leaves it alone. For a positive `r` the rule holds. In physics, masses, lengths
and frequencies are positive: say so, and SymPy simplifies much more.

</details>

### 2.3 Substitution

`subs` replaces a symbol by another symbol, by an expression or by a number:

In [ ]:
expr = x**2 + y**2
print(expr.subs(y, x))
print(expr.subs(y, sp.sin(x)))
print(expr.subs({x: 2, y: 3}))

Expressions are **immutable**: `subs` returns a new expression and `expr` is unchanged.

### 2.4 From strings and to LaTeX

`sympify` parses a string into an expression. `pprint` and `latex` print it:

In [ ]:
expr = sp.sympify('x**2 + 3*x + 2')
print(type(expr))
sp.pprint(expr)
print(sp.latex(sp.sqrt(x**2 + 1) / (x - 1)))

In a notebook, the last expression of a cell is shown rendered in LaTeX.

## 3. Simplification

* `simplify`: tries, *heuristically*, to find the simplest form;
* `expand`, `factor`: products into sums and back;
* `collect`, `cancel`, `trigsimp`: more specific, and more predictable.

In [ ]:
a, b = sp.symbols('a b')
print(sp.simplify(sp.sin(x)**2 + sp.cos(x)**2))
print(sp.expand((a + b)**3))
print(sp.factor(x**3 - 3*x**2 + 3*x - 1))
print(sp.collect(a*x**2 + b*x**2 + a*x, x))
print(sp.cancel((x**2 - 1) / (x - 1)))

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — One third</b></p>

This code should print `True`, and prints `False`. Why? Fix it.

```python
third = sp.simplify((x**2 - 1) / (3*x**2 - 3))
print(third)
print(third == 1/3)
```

</div>

In [ ]:
third = sp.simplify((x**2 - 1) / (3*x**2 - 3))
print(third)
print(third == 1/3)

<details>
<summary><b>Solution</b></summary>

`third` is the exact rational `1/3`, while `1/3` in Python is the float
`0.3333333333333333`: they are different objects. Compare with an exact number,
`third == sp.Rational(1, 3)`, or numerically, `math.isclose(third, 1/3)`.

The same trap appears when building expressions: `1/2*x` is `0.5*x`, a float;
`sp.Rational(1, 2)*x` or `x/2` keep it exact.

</details>

## 4. Calculus

### 4.1 Derivatives

`diff(expr, x)` differentiates with respect to `x`; `diff(expr, x, n)` gives the
$n$-th derivative.

In [ ]:
expr = x * sp.sin(x**2) + 1
print(sp.diff(expr, x))
print(sp.diff(expr, y))
print([sp.diff(x**5, x, n) for n in range(1, 7)])

### 4.2 Integrals

`integrate(expr, x)` gives an antiderivative (without the constant);
`integrate(expr, (x, lower, upper))` a definite integral. `sp.oo` is $\infty$.

In [ ]:
print(sp.integrate(x**2 + x + 1, x))
print(sp.integrate(sp.exp(-x**2), (x, -sp.oo, sp.oo)))

Multiple integrals: one tuple per variable. The volume of a sphere of radius $R$, in
spherical coordinates:

In [ ]:
R, rho, theta, phi = sp.symbols('R rho theta phi', positive=True)
sp.integrate(rho**2 * sp.sin(theta), (rho, 0, R), (theta, 0, sp.pi), (phi, 0, 2*sp.pi))

### 4.3 Limits and series

`limit(expr, x, x0)` and `series(expr, x, x0, n)`, the Taylor expansion up to (not
including) order $n$:

In [ ]:
sp.limit(sp.sin(x) / x, x, 0)

The relativistic energy, $E = mc^2/\sqrt{1 - v^2/c^2}$, at small velocity:

* rest energy, $mc^2$;
* kinetic energy, $mv^2/2$;
* first correction, $3mv^4/8c^2$.

In [ ]:
m, v, c = sp.symbols('m v c', positive=True)
E = m * c**2 / sp.sqrt(1 - v**2 / c**2)
sp.series(E, v, 0, 6)

### 4.4 Fourier series and integral transforms

`fourier_series`, `fourier_transform`, `laplace_transform`, …

A step function in $(-\pi, \pi)$, and the first terms of its Fourier series.
`Piecewise((value, condition), ...)` defines a function by pieces; the first true
condition wins:

In [ ]:
step = sp.Piecewise((0, x < 0), (1, True))
sp.fourier_series(step, (x, -sp.pi, sp.pi)).truncate(4)

In [ ]:
k = sp.symbols('k', real=True)
sp.fourier_transform(sp.exp(-x**2), x, k)

Check the convention in the documentation: SymPy uses $e^{-2\pi i k x}$, not
$e^{-ikx}$.

## 5. Solvers

### 5.1 solve and nsolve

An expression passed to a solver is understood as `expr = 0`; use `sp.Eq(lhs, rhs)`
for an equation.

* `solve`: exact solutions, if it can find them;
* `nsolve`: a numerical root, starting from an initial guess.

In [ ]:
print(sp.solve(x**2 - y**2, y))
print(sp.solve(x**2 - 2*x + 1, x))

$x^2 - 2x + 1 = (x-1)^2$ has a double root, but `solve` returns it once;
`sp.roots(x**2 - 2*x + 1)` gives the multiplicities, `{1: 2}`.

In [ ]:
expr = sp.log(x) - x + 6
root = sp.nsolve(expr, x, 5)          # 5 is the initial guess
print(root, expr.subs(x, root))

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Warm-up</b></p>

1. Build $f = x^2 + 2x - 3$ and find its value at $x = 5$.
2. Factor $x^3 - 3x^2 + 2x - 6$ and find its roots.
3. Integrate $\sin x \cos x$: an antiderivative, and the definite integral in
   $(-\pi, \pi)$.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
f = x**2 + 2*x - 3
print(f.subs(x, 5))                                  # 32

p = x**3 - 3*x**2 + 2*x - 6
print(sp.factor(p))                                  # (x - 3)*(x**2 + 2)
print(sp.solve(p, x))                                # [3, -sqrt(2)*I, sqrt(2)*I]

g = sp.sin(x) * sp.cos(x)
print(sp.integrate(g, x))                            # sin(x)**2/2
print(sp.integrate(g, (x, -sp.pi, sp.pi)))           # 0
```

The antiderivative is defined up to a constant: $-\cos^2 x/2$ is also correct. The
definite integral is zero because the integrand is odd.

</details>

### 5.2 Systems of equations

`linsolve` for linear systems, `nonlinsolve` for the rest. Both return a set of
tuples.

$$
\begin{cases}
x - y = 1 \\
x + y = 2
\end{cases}
\qquad\qquad
\begin{cases}
a^2 + a = 0 \\
a - b = 0
\end{cases}
$$

In [ ]:
print(sp.linsolve([x - y - 1, x + y - 2], (x, y)))
print(sp.nonlinsolve([a**2 + a, a - b], (a, b)))

### 5.3 Differential equations

`dsolve` solves ordinary differential equations. The unknown is a `Function`. First,
$f(x) - f'(x) = \sin x$:

In [ ]:
f = sp.Function('f')
sp.dsolve(f(x) - f(x).diff(x) - sp.sin(x), f(x))

The damped harmonic oscillator, $\ddot{u} + 2\gamma\dot{u} + \omega_0^2 u = 0$. The
general solution has two constants, `C1` and `C2`. Time is declared `nonnegative=True`,
another assumption ($t \geq 0$):

In [ ]:
t = sp.symbols('t', nonnegative=True)
gamma, omega0 = sp.symbols('gamma omega_0', positive=True)
u = sp.Function('u')
ode = sp.Eq(u(t).diff(t, 2) + 2*gamma*u(t).diff(t) + omega0**2 * u(t), 0)
sp.dsolve(ode)

With numbers ($\gamma = 1/10$, $\omega_0 = 1$) and **initial conditions**
($u(0) = 1$, $\dot{u}(0) = 0$), passed with `ics`:

In [ ]:
ode1 = ode.subs({gamma: sp.Rational(1, 10), omega0: 1})
sol = sp.dsolve(ode1, ics={u(0): 1, u(t).diff(t).subs(t, 0): 0})
sol

## 6. Matrices

`sp.Matrix` holds exact entries, or symbols. Operators work as in linear algebra:
`*` is the matrix product.

In [ ]:
M = sp.Matrix([[1, 2, 3], [2, 3, 1], [3, 0, 1]])
M2 = sp.Matrix([[0, 4, 3], [2, 0, 1], [1, 0, 1]])
M.inv()

In [ ]:
print(M + M2)
print(M * M2)
print(M.det(), M.rank())

A matrix has many methods; filter `dir` to find them:

In [ ]:
print([name for name in dir(M) if name.startswith(('det', 'eigen', 'inv', 'rank'))])

With symbols: the eigenvalues of a $2\times 2$ symmetric matrix, as in two coupled
oscillators.

In [ ]:
w, kappa = sp.symbols('omega kappa', positive=True)
K = sp.Matrix([[w**2 + kappa, -kappa], [-kappa, w**2 + kappa]])
K.eigenvals()

The squared frequencies of the two normal modes: in phase, $\omega^2$; in antiphase,
$\omega^2 + 2\kappa$.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Linear algebra</b></p>

1. Solve the system $2x + 3y = 5$, $3x - 2y = 7$.
2. For $A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$, find its inverse, its
   determinant and its eigenvalues. Check that the product of the eigenvalues is the
   determinant.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
print(sp.linsolve([2*x + 3*y - 5, 3*x - 2*y - 7], (x, y)))   # {(31/13, 1/13)}

A = sp.Matrix([[1, 2], [3, 4]])
print(A.inv())                       # Matrix([[-2, 1], [3/2, -1/2]])
print(A.det())                       # -2
eigen = A.eigenvals()                # {5/2 - sqrt(33)/2: 1, 5/2 + sqrt(33)/2: 1}
print(eigen)
print(sp.expand(sp.prod(eigen.keys())) == A.det())   # True
```

`eigenvals` returns a dict `{eigenvalue: multiplicity}`. The product
$(5/2)^2 - 33/4 = -2$ needs `expand` to collapse to an integer.

</details>

## 7. Geometry

`sympy.geometry` handles points, lines, circles and polygons, in 2D and 3D, exactly.

In [ ]:
from sympy.geometry import Point, Line, Circle

print(Point.is_collinear(Point(0, 0), Point(2, 2), Point(4, 4)))
print(Point.is_collinear(Point(0, 0, 0), Point(2, 2, 2), Point(4, 4, 0)))
print(Line(Point(0, 5), Point(5, 0)).intersection(Line(Point(0, 0), Point(5, 5))))

The 3D points are not collinear: $(4, 4, 0)$ is off the line through the first two.
`intersection` returns a list: here one point; empty for parallel lines.

In [ ]:
z = sp.symbols('z')
print(Point(x, y, z).distance(Point(0, 0, 0)))
print(Circle(Point(0, 0), 5).equation(x, y))

## 8. From symbols to numbers

### 8.1 evalf

`evalf(n)` evaluates an expression with $n$ significant digits; `subs` gives the values
of the symbols.

In [ ]:
print(sp.pi.evalf(30))
print((a / b).evalf(20, subs={a: 100, b: 3}))

### 8.2 lambdify

`evalf` is slow: it works with SymPy objects. `lambdify` turns an expression into an
ordinary Python function that uses **NumPy**: fast, and it accepts arrays.

In [ ]:
u_t = sp.lambdify(t, sol.rhs)          # sol: the damped oscillator of section 5.3
ts = np.linspace(0, 40, 400)

plt.plot(ts, u_t(ts))
plt.xlabel('t'); plt.ylabel('u(t)')
plt.show()

The usual path: derive the formula with SymPy, then compute with NumPy.

### 8.3 SymPy plots

SymPy can also plot expressions directly, with matplotlib behind:

In [ ]:
p = sp.plot(sp.sin(x), sp.cos(x), (x, -sp.pi, sp.pi), xlabel='x', ylabel='f(x)')

In [ ]:
from sympy.plotting import plot3d

p = plot3d(x**2 + y**2, (x, -10, 10), (y, -10, 10))

Assign the plot (`p = ...`): otherwise the notebook also prints the `Plot` object.

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Symbolic or numerical?</b></p>

For each problem, would you use SymPy, NumPy/SciPy, or both?

1. The Taylor expansion of a potential around its minimum.
2. The trajectory of a pendulum at large amplitude.
3. The integral of a histogram of $10^6$ events.
4. The eigenvalues of a $1000 \times 1000$ matrix of measured couplings.

</div>

<details>
<summary><b>Solution</b></summary>

1. SymPy: `series` gives the exact coefficients (the frequency of small oscillations).
2. Both: SymPy to write the equations (and check the small-angle limit), SciPy
   (`solve_ivp`) to integrate them; the large-amplitude pendulum has no elementary
   solution.
3. NumPy: the data are numbers, there is nothing symbolic.
4. NumPy (`np.linalg.eigvalsh` for symmetric matrices; otherwise `eigvals`): symbolic eigenvalues are only feasible for small
   matrices, and here the entries are floats anyway.

Symbolic computing is exact but slow, and expressions can grow very fast. Use it for the
derivation; compute with numbers.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A chain of rings</b></p>

A chain is made of five steel rings. Each ring is a solid **torus**: its tube, of radius
$a$, goes around a circle of radius $R$.

Use polar coordinates $(r, \theta)$ in the cross-section of the tube, plus the angle
$\phi$ around the axis. A point is at distance $R + r\cos\theta$ from the axis, and the
volume element is $r\,(R + r\cos\theta)\,dr\,d\theta\,d\phi$.

</div>

1. Compute the volume $V$ of a torus by integration.
2. Compute its moment of inertia about its symmetry axis, $I = \int \rho\, d^2\, dV$
   ($d$: distance to the axis), with constant density $\rho$. Write it in terms of the
   mass $M$.
3. With $R = 2$ cm, $a = 0.4$ cm and $\rho = 7.87$ g/cm$^3$, what is the mass of the
   chain?
4. Plot $I / (M R^2)$ as a function of $a/R$ in $[0, 1]$.

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
Rt, at, rt = sp.symbols('R a r', positive=True)       # torus: R, a; coordinate r
dens = sp.symbols('rho', positive=True)                # density
dist = Rt + rt * sp.cos(theta)                         # distance to the axis
dV = rt * dist                                         # volume element (without the d's)
lims = ((rt, 0, at), (theta, 0, 2*sp.pi), (phi, 0, 2*sp.pi))

V = sp.integrate(dV, *lims)
print(V)                                               # 2*pi**2*R*a**2

Mt = dens * V
It = sp.integrate(dens * dist**2 * dV, *lims)
print(sp.factor(It / Mt))                              # (4*R**2 + 3*a**2)/4

mass = 5 * Mt.subs({Rt: 2, at: sp.Rational(2, 5), dens: sp.Rational(787, 100)})
print(mass, '=', mass.evalf(4), 'g')                   # about 249 g

q = sp.symbols('q', positive=True)                     # q = a/R
ratio = sp.simplify((It / (Mt * Rt**2)).subs(at, q * Rt))
p = sp.plot(ratio, (q, 0, 1), xlabel='a/R', ylabel='I/(M R^2)')
```

$V = 2\pi^2 R a^2$ (Pappus: the area of the section, $\pi a^2$, times the length of
the circle, $2\pi R$), and $I = M\,(R^2 + \tfrac{3}{4}a^2)$. For a thin ring,
$a \ll R$, we recover $I = MR^2$. The volumes of the rings do not overlap, so the mass
of the chain is five times that of one ring.

</details>

***[+] Lookout.*** SymPy goes much further: `sympy.physics` (classical mechanics with Lagrangians and
Kane's method, quantum mechanics, units), `sympy.stats` (random variables), and code
generation (`sympy.codegen`, `sp.ccode`) to turn a derived formula into C or Fortran.

## 9. Summary

* SymPy computes **exactly**: symbols, rationals, $\pi$, $\sqrt{2}$.
* Declare **assumptions** (`positive=True`): they allow more simplifications.
* Calculus, series, solvers and `dsolve` reproduce the work on paper.
* `sp.Matrix`: exact linear algebra, also with symbolic entries.
* `sympy.geometry`: points, lines and circles, exact intersections and distances.
* Derive with SymPy, compute with NumPy: **lambdify**.

<details>
<summary><b>[i] Cheat sheet — SymPy</b></summary>

| | |
|:--|:--|
| `sp.symbols('x y', positive=True)` | symbols, with assumptions |
| `sp.Rational(1, 3)`, `sp.pi`, `sp.oo`, `sp.I` | exact numbers |
| `expr.subs({x: 2})`, `sp.sympify('x**2')` | substitute; parse a string |
| `simplify`, `expand`, `factor`, `collect`, `cancel` | simplification |
| `sp.diff(e, x, n)`, `sp.integrate(e, (x, a, b))` | derivative, integral |
| `sp.limit(e, x, x0)`, `sp.series(e, x, x0, n)` | limit, Taylor series |
| `sp.solve(e, x)`, `sp.nsolve(e, x, x0)` | exact, numerical roots |
| `sp.linsolve`, `sp.nonlinsolve` | systems |
| `sp.dsolve(ode, f(x), ics={...})` | ordinary differential equations |
| `sp.Matrix`, `.inv()`, `.det()`, `.eigenvals()` | linear algebra |
| `e.evalf(n)`, `sp.lambdify(x, e)` | to numbers; to a NumPy function |
| `Point`, `Line`, `Circle`, `.intersection()` | geometry |
| `sp.plot`, `plot3d`, `sp.latex`, `sp.pprint` | plot, print |

</details>

### Check yourself

1. What is the difference between `math.sqrt(8)` and `sp.sqrt(8)`?
2. Why does SymPy not simplify $\sqrt{x^2}$ to $x$? How do you make SymPy simplify it?
3. Why is `sp.integrate(x**2, (x, 0, 1)) == 1/3` false?
4. When do you use `solve`, and when `nsolve`?
5. How do you give initial conditions to `dsolve`?
6. What does `M.eigenvals()` return for a `sp.Matrix`?
7. What does `Line.intersection` return?
8. What does `lambdify` do, and why is it useful?

<details>
<summary><b>Solutions</b></summary>

1. `math.sqrt(8)` is a float, `2.828...`; `sp.sqrt(8)` is the exact number $2\sqrt{2}$.
2. Because it is false for negative or complex $x$. Declare the symbol
   `positive=True` (or `real=True`, which gives $|x|$).
3. The integral is the exact rational `1/3`; Python's `1/3` is a float. Compare with
   `sp.Rational(1, 3)`.
4. `solve` when an exact solution exists and is wanted; `nsolve` for a numerical root,
   from an initial guess, when there is no closed form (e.g. $\log x = x - 6$).
5. With `ics`, a dict: `ics={f(0): 1, f(x).diff(x).subs(x, 0): 0}`.
6. A dict `{eigenvalue: multiplicity}`, with exact (possibly symbolic) eigenvalues.
7. A list: the intersection points (one for two crossing lines, none if parallel).
8. It turns an expression into a Python function that uses NumPy: fast evaluation on
   arrays, for plots and numerical work.

</details>